# Evaluation and Tuning

This notebook is to document evaluation trials analysis and log changes after each tuning approach

## Datasets:

- **evaluation.csv**: 100 Statistics theory and applications questions with ground truths and ground contexts


## Evaluation approach:

- **LLM-as-a-judge**: Using LLMs to justify the answer and give evaluation metrics. The metrics generated by this method are: correctness and faithfulness

- **Framework**: **LangSmith** platform to monitor and record experiments.

## Evaluation metrics used:

### Correctness

- **What it measures:** Internal logical consistency.

- **The Definition:** Does the model's reasoning logically support the final answer it gave to the user's question?

- **How it works:** It looks only at the Question, the Answer, and the Reasoning. It does not check if the facts are true in the real world; it only checks if the model's thought process makes logical sense from point A to point B.

### Faithfulness

- **What it measures:** Groundedness and absence of hallucinations.

- **The Definition:** Is every single claim in the generated answer entirely backed up by the retrieved documents?

- **How it works:** It looks at the Answer and the Retrieved Context. If the model brings in outside knowledge that isn't in your text chunks—even if that outside knowledge is technically true—it fails the faithfulness check.

### Notes

- All LLMs in used were hosted locally via Ollama. Hence, I can evaluate the system expansively without worrying about cost. However, the latency is high since my machine is not as powerful as corporate systems :(

## Experiment: glossy-root-69 (Baseline)

### Dataset: 'evaluations.csv'
### RAG schema: Ingestion -> Chunking (size = 200, overlap =  100) -> BM_25 (0.7) + ChromaDB Retriever (0.3) -> Google Gemma 3 (Ollama)
### Metris: Correctness (DeepSeek-R1)

In [1]:
from langsmith import Client
from dotenv import load_dotenv
from src.utils import beautify_langsmith_stats

load_dotenv()

client = Client()
exp = client.get_experiment_results(name='glossy-root-69')
print(beautify_langsmith_stats(exp))

### Evaluation Summary
* **Feedback: valid_reasoning**
        - **Average Score:** 76.3% (0.763)
        - **Standard Deviation:** 0.425
        - **Evaluated Runs:** 452 (Errors: 0)
* **Run Performance & Telemetry**
        - **Total Runs:** 453
        - **Latency:** 1.27s (Median) | 3.23s (99th Percentile)
        - **Token Usage:** 327,427 Total (269,282 prompt / 58,145 completion)
        - **Last Run:** March 14, 2026 at 06:13 AM


dict_values([{'check_faithfulness': {'n': 309, 'avg': 0.9255058252427184, 'stdev': 0.1841455776077012, 'errors': 0, 'values': {}, 'type': 'primary', 'contains_thread_feedback': False}, 'valid_reasoning': {'n': 309, 'avg': 0.8317152103559871, 'stdev': 0.37411899072151267, 'errors': 0, 'values': {}, 'type': 'primary', 'contains_thread_feedback': False}}, {'run_count': 310, 'latency_p50': datetime.timedelta(seconds=3, microseconds=566000), 'latency_p99': datetime.timedelta(seconds=5, microseconds=177990), 'total_tokens': 223642, 'prompt_tokens': 184506, 'completion_tokens': 39136, 'last_run_start_time': datetime.datetime(2026, 3, 14, 8, 27, 7, 968333), 'run_facets': None, 'total_cost': Decimal('0.0'), 'prompt_cost': Decimal('0.0'), 'completion_cost': Decimal('0.0'), 'first_token_p50': datetime.timedelta(seconds=2, microseconds=676000), 'first_token_p99': datetime.timedelta(seconds=2, microseconds=863470), 'error_rate': 0.0}, <generator object Client.get_experiment_results.<locals>._get_ex

## Analysis:
- By using

## Experiment: dependable-loss-22

### Dataset: 'evaluations.csv'
### RAG schema: Ingestion -> Chunking (size = 200, overlap =  100) -> BM_25 (0.7) + ChromaDB Retriever (0.3) -> Google Gemma 3 (Ollama)
### Metris: Correctness (DeepSeek-R1), Faithfulness (Llama3.1)

In [1]:
from langsmith import Client
from dotenv import load_dotenv
from src.utils import beautify_langsmith_stats

load_dotenv()

client = Client()
exp = client.get_experiment_results(name='dependable-loss-22')
print(beautify_langsmith_stats(exp))

### Evaluation Summary

* **Feedback: check_faithfulness**
        - **Average Score:** 92.6% (0.926)
        - **Standard Deviation:** 0.184
        - **Evaluated Runs:** 309 (Errors: 0)
* **Feedback: valid_reasoning**
        - **Average Score:** 83.2% (0.832)
        - **Standard Deviation:** 0.374
        - **Evaluated Runs:** 309 (Errors: 0)
* **Run Performance & Telemetry**
        - **Total Runs:** 310
        - **Latency:** 3.57s (Median) | 5.18s (99th Percentile)
        - **Token Usage:** 223,642 Total (184,506 prompt / 39,136 completion)
        - **Last Run:** March 14, 2026 at 08:27 AM


## Experiment: new-stone-19

### Dataset: 'evaluations.csv'
### RAG schema: Ingestion -> Chunking (size = 250, overlap =  100) -> BM_25 (0.7) + ChromaDB Retriever (0.3) -> Google Gemma 3 (Ollama)
### Metris: Correctness (DeepSeek-R1), Faithfulness (Llama3.1)

In [2]:
from langsmith import Client
from dotenv import load_dotenv
from src.utils import beautify_langsmith_stats

load_dotenv()

client = Client()
exp = client.get_experiment_results(name='new-stone-19')
print(beautify_langsmith_stats(exp))

### Evaluation Summary
* **Feedback: check_faithfulness**
        - **Average Score:** 87.8% (0.878)
        - **Standard Deviation:** 0.223
        - **Evaluated Runs:** 144 (Errors: 0)
* **Feedback: valid_reasoning**
        - **Average Score:** 79.9% (0.799)
        - **Standard Deviation:** 0.401
        - **Evaluated Runs:** 144 (Errors: 0)

* **Run Performance & Telemetry**
        - **Total Runs:** 145
        - **Latency:** 3.47s (Median) | 5.17s (99th Percentile)
        - **Token Usage:** 117,763 Total (98,834 prompt / 18,929 completion)
        - **Last Run:** March 14, 2026 at 08:32 PM


# Conclusion:

- Using 2 LLMs to evaluate increased the latency of eacg evaluation. However the system performed well with the set up (chunk size: 200, chunk overlap: 100).
- Both correctness and faithfulness decreased after increasing chunk size and chunk overlap

# Current Optimized setup

#### RAG schema: Ingestion -> Chunking (size = 200, overlap =  100) -> BM_25 (0.7) + ChromaDB Retriever (0.3) -> Google Gemma 3 (Ollama)


In [6]:
from src.pipeline import pipeline_enforced
from langchain_ollama import ChatOllama

model = ChatOllama(model='gemma3', temperature=0)

chain = pipeline_enforced(model)
# chain.invoke('What are assumptions of arima?')


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10283.09it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
chain.invoke("What are the assumptions of ARIMA?")

RAGResponse(answer='The ARIMA model (p,d,q) has several key assumptions. Primarily, it assumes that the time series data follows an ARMA(p,q) process, which means it’s a stationary and invertible ARMA model (6.1.1) where the parameters φ and θ are estimated (Box, 238). Additionally, the model assumes that the time series is free of the impact of outliers and follows an additive process (Box, 516). However, the specific assumptions depend on the values of p, d, and q. For example, root tests are considered for mixed ARIMA models to account for serial correlation (Yap and Reinsel, 1995; Shin). If the ‘d’ value is greater than 0, it implies that the time series has been differenced to achieve stationarity, and the assumptions of the differenced series must also be met (Devore, 142).', citations=[Citation(author='Box', title='Time Series Analysis', page='238', year=1986), Citation(author='Box', title='Time Series Analysis', page='516', year=1986), Citation(author='Yap and Reinsel', title='